In [1]:
import os
import certifi

from dotenv import load_dotenv
from langchain_openrouter import ChatOpenRouter
from langchain_tavily import TavilySearch
from langsmith import Client
from langchain.tools import tool
import requests

In [2]:
from langchain.agents import create_agent

In [3]:
load_dotenv()

client = Client()

prompt = client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)
extracted_template = prompt.template


In [4]:
OPENROUTER_API_KEY=os.getenv("OPENROUTER_API_KEY")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY=os.getenv("WEATHERSTACK_API_KEY")

In [10]:
search_tool = TavilySearch(max_results= 3, topic="general",)

In [9]:
@tool()
def get_weather_data(city: str) -> str :
    """
    Fetch current weather information for a city.

    Args:
        It takes city as arg and it's a string

    Returns:
        It returns Str - Current weather information
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )


In [11]:
search_tool.invoke({"query": "What happened at the last wimbledon"})

{'query': 'What happened at the last wimbledon',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.the-independent.com/topic/wimbledon',
   'title': 'Wimbledon - latest news, breaking stories and comment - The Independent',
   'content': "The body was found in a wooded area of Borstal Recreation Ground in Rochester\n\n## Two men charged with murder after body found dumped in woodland\n\n<p>Alexander Zverev plays defending champion Jannik Sinner in his first Wimbledon final </p>\n\n## Why a two-week transformation poses Wimbledon final’s biggest question\n\n<p>Alexander Zverev plays in his first Wimbledon final against Jannik Sinner on Sunday </p>\n\n## Zverev and the allegations casting a cloud over Wimbledon final [...] <p>Arthur Fery has jumped up the rankings to world No 36 and British No 1</p>\n\n## Fery wins first match since stunning run to Wimbledon semi-finals\n\n## Fery wins first match since stunning run to Wimbledon semi-finals\n

In [12]:
llm = ChatOpenRouter(
    model = "inclusionai/ling-3.0-flash-vl:free",
    temperature=0,
)

In [13]:
llm.invoke("Tell me a joke")

AIMessage(content="Why don't scientists trust atoms?\n\nBecause they make up everything! 😄", additional_kwargs={'reasoning_content': "The user is asking for a joke. This is a simple, lighthearted request. I should provide a clean, funny joke that works well in English. Let me think of a classic joke that's universally amusing.\n\nA good option: a classic setup-punchline joke. Let me go with something light and fun.", 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': "The user is asking for a joke. This is a simple, lighthearted request. I should provide a clean, funny joke that works well in English. Let me think of a classic joke that's universally amusing.\n\nA good option: a classic setup-punchline joke. Let me go with something light and fun."}]}, response_metadata={'model_name': 'inclusionai/ling-3.0-flash-vl:free', 'id': 'gen-1789404482-so94OavSSkT49SVz1Rmb', 'created': 1789404482, 'object': 'chat.completion', 'finish_reason': 'stop', 'logp

In [14]:
tools = [search_tool, get_weather_data]

In [16]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=extracted_template
)


In [17]:
result = agent.invoke({"messages": [{"role": "user", "content": "What is the capital of Ilorin and what is teh current weather condition"}]})


In [18]:
result

{'messages': [HumanMessage(content='What is the capital of Ilorin and what is teh current weather condition', additional_kwargs={}, response_metadata={}, id='8f21d528-a443-4b12-a908-b8c1204f7922'),
  AIMessage(content='\n', additional_kwargs={'reasoning_content': "The user is asking two things:\n1. What is the capital of Ilorin? - Ilorin is a city in Nigeria, it's the capital of Kwara State. Let me search to confirm.\n2. What is the current weather condition in Ilorin?\n\nLet me search for both simultaneously.", 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': "The user is asking two things:\n1. What is the capital of Ilorin? - Ilorin is a city in Nigeria, it's the capital of Kwara State. Let me search to confirm.\n2. What is the current weather condition in Ilorin?\n\nLet me search for both simultaneously."}]}, response_metadata={'model_name': 'inclusionai/ling-3.0-flash-vl:free', 'id': 'gen-1789404536-q0SDv9Tm0YJHfqTtwDVR', 'created': 17894045

Failed to refresh cache entry hwchase17/react: Connection error caused failure to GET /commits/hwchase17/react/latest in LangSmith API. Please confirm your internet connection. ConnectTimeout(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /commits/hwchase17/react/latest (Caused by ConnectTimeoutError(<HTTPSConnection(host='api.smith.langchain.com', port=443) at 0x28d4da4f390>, 'Connection to api.smith.langchain.com timed out. (connect timeout=10.0)'))"))
Content-Length: None
API Key: 


In [ ]:
## I Use stream
#
# agent.stream(
#
# )